# Task 1.2 — Assumptions of the Paper

**Paper:** *Analysis of SVM with Indefinite Kernels* (NIPS 2009)  
**Authors:** Yiming Ying, Colin Campbell, Mark Girolami

---

This notebook identifies and analyzes at least 3 key assumptions made by the paper.

---

## Assumption 1: Kernel Matrix Can Be Well-Approximated by a PSD Proxy

**Assumption:**  
The original indefinite kernel matrix $K$ can be meaningfully approximated by a positive semi-definite (PSD) matrix $\tilde{K}$ that preserves the relevant similarity structure for classification.

**Why the method needs it:**  
The entire framework relies on projecting $K$ onto the PSD cone to obtain $K_{+}$. If the PSD proxy does not capture the classification-relevant structure, the resulting SVM will underperform. The method assumes that the negative eigenvalues of $K$ are primarily noise or artifacts, and that the positive eigenspace carries the discriminative information.

**Violation scenario:**  
If the negative eigenvalues of $K$ encode important discriminative information (e.g., in certain non-metric similarity measures where dissimilarity is as informative as similarity), then projecting to PSD will discard critical information, leading to poor classification performance.

**Paper reference:**  
Section 2, Equation (3) — the constraint $\tilde{K} \succeq 0$ with $\|\tilde{K} - K\|_F \leq \rho$.

---

## Assumption 2: Lipschitz Continuous Gradient

**Assumption:**  
The gradient of the objective function $f(\alpha)$ is Lipschitz continuous with constant $L = \|K_{+}\|$ (the spectral norm of the PSD-projected kernel).

**Why the method needs it:**  
The convergence guarantees of both the projected gradient method ($O(1/t)$) and the Nesterov accelerated method ($O(1/t^2)$) require Lipschitz continuity of the gradient. The step size $1/L$ is directly derived from this constant. Without this assumption, the gradient steps may overshoot and the algorithm may diverge.

**Violation scenario:**  
If the kernel matrix has extremely large eigenvalues (e.g., due to poor scaling or outliers), $L$ becomes very large, making the step size $1/L$ extremely small. This leads to very slow convergence in practice, even though the theoretical guarantee still holds. In extreme cases, numerical instability can cause effective violation of the smoothness assumption.

**Paper reference:**  
Section 3, Lemma 2 and Theorem 2 — Lipschitz constant derivation and convergence bound.

---

## Assumption 3: Feasibility of Eigen-Decomposition

**Assumption:**  
The eigen-decomposition of the kernel matrix $K \in \mathbb{R}^{n \times n}$ is computationally feasible, i.e., the dataset size $n$ is small enough that $O(n^3)$ computation and $O(n^2)$ memory are acceptable.

**Why the method needs it:**  
The PSD projection step requires computing $K = Q\Lambda Q^T$ and then forming $K_{+} = Q\Lambda_{+}Q^T$. This is the computational bottleneck of the algorithm. The full kernel matrix must be stored in memory, and the eigen-decomposition must be computed.

**Violation scenario:**  
For datasets with $n > 10{,}000$ samples, storing the $n \times n$ kernel matrix requires prohibitive memory (e.g., $n = 50{,}000$ requires ~18.6 GB for float64). The eigen-decomposition itself becomes extremely slow. This limits the method to small-to-medium datasets in practice.

**Paper reference:**  
Section 3, Proposition 1 — PSD projection via eigen-decomposition; Section 5, Experimental results on datasets of moderate size.

---

## Assumption 4: Projection onto PSD Cone is the Optimal Strategy

**Assumption:**  
Projecting onto the PSD cone (clipping negative eigenvalues to zero) is the best way to handle kernel indefiniteness. The method assumes that the Frobenius norm is the appropriate distance metric for measuring kernel proximity.

**Why the method needs it:**  
The closed-form solution for the inner maximization problem relies on the fact that the nearest PSD matrix in Frobenius norm is obtained by clipping negative eigenvalues. Different norms (e.g., spectral norm, KL-divergence) would lead to different projections and potentially different optimization landscapes.

**Violation scenario:**  
If the kernel indefiniteness is structured (e.g., consistently negative in a specific subspace that corresponds to a meaningful data pattern), the Frobenius-norm projection may not be the most informative. Alternative projections (e.g., "flipping" negative eigenvalues to positive, or using a weighted norm) might preserve more information for specific applications.

**Paper reference:**  
Section 2, constraint set $\mathcal{K}$; Proposition 1 — nearest PSD matrix result.

---

## Summary Table

| # | Assumption | Consequence of Violation |
|---|-----------|-------------------------|
| 1 | PSD proxy preserves classification structure | Loss of discriminative information |
| 2 | Lipschitz continuous gradient | Slow convergence or divergence |
| 3 | Eigen-decomposition is computationally feasible | Scalability bottleneck for large $n$ |
| 4 | Frobenius-norm PSD projection is optimal | Suboptimal kernel approximation |